In [1]:
! uv pip install vllm

Using Python 3.12.12 environment at: /usr
Resolved 173 packages in 1.09s                                       
Prepared 53 packages in 14.19s                                           
Uninstalled 7 packages in 836ms
Installed 53 packages in 324ms                              
 + anthropic==0.86.0
 + apache-tvm-ffi==0.1.9
 + astor==0.8.1
 + blake3==1.0.8
 + cbor2==5.8.0
 + compressed-tensors==0.13.0
 + depyf==0.20.0
 + diskcache==5.6.3
 + dnspython==2.8.0
 + email-validator==2.3.0
 + fastapi-cli==0.0.24
 + fastapi-cloud-cli==0.15.0
 + fastar==0.9.0
 + flashinfer-python==0.6.6
 + gguf==0.18.0
 - huggingface-hub==1.7.1
 + huggingface-hub==0.36.2
 + ijson==3.5.0
 + interegular==0.3.3
 + jmespath==1.1.0
 - lark==1.3.1
 + lark==1.2.2
 + llguidance==1.3.0
 - llvmlite==0.43.0
 + llvmlite==0.44.0
 + lm-format-enforcer==0.11.3
 + loguru==0.7.3
 + mistral-common==1.10.0
 + model-hosting-container-standards==0.1.14
 + msgspec==0.20.0
 + ninja==1.13.0
 - numba==0.60.0
 + numba==0.61.2
 + nvidia-c

# 1st Steps

## Load Model

In [2]:
from vllm import LLM, SamplingParams

model_id = "Qwen/Qwen3-1.7B-Base"
llm = LLM(model_id)

INFO 03-22 14:30:55 [utils.py:233] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen3-1.7B-Base'}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

INFO 03-22 14:31:31 [model.py:533] Resolved architecture: Qwen3ForCausalLM
WARNING 03-22 14:31:31 [model.py:1867] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 03-22 14:31:31 [model.py:1920] Casting torch.bfloat16 to torch.float16.
INFO 03-22 14:31:31 [model.py:1582] Using max model len 32768
INFO 03-22 14:31:31 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-22 14:31:31 [vllm.py:754] Asynchronous scheduling is enabled.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

WARNING 03-22 14:31:33 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 03-22 14:36:07 [llm.py:391] Supported tasks: ['generate']


### Tests :)

In [30]:
sampling_params = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=128)
prompts = [
    "Hello my name is",
    "what is the capital city of ",
    "explain transformer architecture:",
]
outputs = llm.generate(prompts, sampling_params=sampling_params)

Rendering prompts:   0%|          | 0/3 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/3 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [ ]:
for out in outputs:
    print(f"prompt: {out.prompt}")
    print(f"response: {out.outputs[0].text}")
    print("=" * 30)

prompt: Hello my name is
response:  Josh and I am a 19-year-old sophomore in college. I am from the United States of America. I am studying computer science in an engineering college. I am interested in the field of technology and how it can be used to improve our lives. My academic interests include computer science, mathematics, and engineering. I am also interested in computer programming and web development. I am looking forward to learning more about the field and exploring how technology can impact society. How can I get started in the field of technology and improve my skills? I am a 19-year-old sophomore in college majoring in computer science. i am looking
prompt: what is the capital city of 
response:  India ?

The capital city of India is **New Delhi**. It is the largest city in India and serves as the political, economic, and cultural center of the country.
prompt: explain transformer architecture:
response:  it has several layers with different types of modules, such as co

## Load Dataset

In [3]:
from datasets import load_dataset

In [4]:
datadataset_name = "HuggingFaceH4/MATH-500"
split = "test"
num_examples = 20
columns = ["problem", "answer"]
dataset = load_dataset(
    datadataset_name, split=f"{split}[:{num_examples}]"
).select_columns(columns)
dataset

README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['problem', 'answer'],
    num_rows: 20
})

In [5]:
dataset[:1]

{'problem': ['Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$'],
 'answer': ['\\left( 3, \\frac{\\pi}{2} \\right)']}

## Format Prompt

In [5]:
def format_prompt(prompt: str, use_cot: bool = False) -> str:
    cot = "\nExplain step by step." if use_cot else ""
    return (
        "You are a helpful math assistant.\n" #personality
        "Answer the question and write the final result on a newline as:\n" # rules ...
        "\\boxed{ANSWER}\n\n"
        f"Question:\n{prompt}\n"
        f"{cot}\n\n"
        "Answer:\n"
    )

## Get Responses

In [6]:
def get_response(
    prompts: list[str], llm: LLM, sampling_params: SamplingParams = None
) -> list[str]:
    outputs = llm.generate(prompts, sampling_params=sampling_params)
    responses = [out.outputs[0].text.strip() for out in outputs]
    return responses

## Extract answer

In [7]:
# extracting the final answer box from the generated answer
def extract_final_answer_box(answer: str) -> str | None:
    box_start_idx = answer.rfind(r"\boxed")
    if box_start_idx == -1:
        return None
    current_idx = box_start_idx + len("r\boxed")
    while current_idx < len(answer) and answer[current_idx].isspace():
        current_idx += 1
    if current_idx == len(answer) or answer[current_idx] != "{":
        return None

    current_idx += 1
    brace_depth = 1  #'we opened {'
    content_start_idx = current_idx

    while current_idx < len(answer) and brace_depth > 0:
        if answer[current_idx] == "{":
            brace_depth += 1
        elif answer[current_idx] == "}":
            brace_depth -= 1
        current_idx += 1
    if brace_depth != 0:
        return None
    return answer[content_start_idx : current_idx - 1]

In [8]:
import re

RE_NUMBER = re.compile(r"-?(?:\d+/\d+|\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)")


def extract_final_answer(answer, fallback="number_then_full"):
    box = extract_final_answer_box(answer)
    result = box or ""
    if result:
        return result
    if fallback:
        numbers = RE_NUMBER.findall(answer)
        if numbers:
            result = numbers[-1]
        elif fallback == "number_then_full":
            result = answer
    return result

In [ ]:
from IPython.display import Math
from IPython.display import display

extracted_response = extract_final_answer(response)
display(Math(extracted_response))

### Utils Function (Math Answer Normalization)

In [9]:
LATEX_FIXES = [
    (r"\\left\s*", ""),
    (r"\\right\s*", ""),
    (r"\\,|\\!|\\;|\\:", ""),
    (r"\\cdot", "*"),
    (r"\u00B7|\u00D7", "*"),
    (r"\\\^\\circ", ""),
    (r"\\dfrac", r"\\frac"),
    (r"\\tfrac", r"\\frac"),
    (r"°", ""),
]
RE_SPECIAL = re.compile(r"<\|[^>]+?\|>")


def normalize_text(text):
    if not text:
        return ""
    text = RE_SPECIAL.sub("", text).strip()
    text = re.sub(r"\^\s*\{\s*\\circ\s*\}", "", text)
    text = re.sub(r"\^\s*\\circ", "", text)
    text = text.replace("°", "")
    match = re.match(r"^\\text\{(?P<x>.+?)\}$", text)
    if match:
        text = match.group("x")
    text = re.sub(r"\\\(|\\\)|\\\[|\\\]", "", text)
    for pat, rep in LATEX_FIXES:
        text = re.sub(pat, rep, text)

    text = text.replace("\\%", "%").replace("$", "").replace("%", "")
    text = re.sub(
        r"\\sqrt\s*\{([^}]*)\}",
        lambda match: f"sqrt({match.group(1)})",
        text,
    )
    text = re.sub(
        r"\\sqrt\s+([^\\\s{}]+)",
        lambda match: f"sqrt({match.group(1)})",
        text,
    )

    text = re.sub(
        r"\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}",
        lambda match: f"({match.group(1)})/({match.group(2)})",
        text,
    )
    text = re.sub(
        r"\\frac\s+([^\s{}]+)\s+([^\s{}]+)",
        lambda match: f"({match.group(1)})/({match.group(2)})",
        text,
    )

    text = text.replace("^", "**")
    text = re.sub(
        r"(?<=\d)\s+(\d+/\d+)",
        lambda match: "+" + match.group(1),
        text,
    )

    text = re.sub(
        r"(?<=\d),(?=\d\d\d(\D|$))",
        "",
        text,
    )
    return text.replace("{", "").replace("}", "").strip().lower()

In [10]:
# parser number: from string number to number
from sympy.parsing import sympy_parser as spp
from sympy.core.sympify import SympifyError
from tokenize import TokenError


def sympy_parser(expr):
    try:
        return spp.parse_expr(
            expr,
            transformations=(
                *spp.standard_transformations,
                spp.implicit_multiplication_application,
            ),
            evaluate=True,
        )
    except (SympifyError, SyntaxError, TypeError, IndexError, TokenError):
        return None

In [ ]:
n = normalize_text(r"\( \frac{1}{2} + \frac{1}{6} \)")
print(n)
sympy_parser(r"((1)/(2) + (1)/(6))")

(1)/(2) + (1)/(6)


2/3

In [11]:
def split_into_parts(exp):
    if exp[0] in "[(" and exp[-1] in ")]" and "," in exp[1:-1]:
        exp = exp[1:-1].split(",")
        return [x.strip() for x in exp]
    return [exp] if exp else []

## Answer Verrifier

In [12]:
def answer_verifier(prediction, truth):
    if prediction == truth:
        return True
    prediction, truth = sympy_parser(prediction), sympy_parser(truth)
    if prediction and truth:
        try:
            if prediction - truth == 0:
                return True
        except Exception:
            pass
    return False


In [13]:
def grad_function(prediction, gtruth):
    results = [False]
    if prediction and gtruth:
        prediction = normalize_text(prediction)
        gtruth = normalize_text(gtruth)
        prediction = split_into_parts(prediction)
        gtruth = split_into_parts(gtruth)
        if prediction and gtruth and (len(gtruth) == len(prediction)):
            results = [answer_verifier(x, y) for x, y in zip(prediction, gtruth)]
    if all(results):
        return True
    return False

## Caclualate_accuracy

In [14]:
def calculate_accuracy(dataset):
    accuracy = sum(dataset["is_correct"]) / len(dataset)
    return accuracy

## Evaluation Loop

In [15]:
import time
from datasets import Dataset


def evaluate(
    model: LLM,
    dataset: Dataset,
    batch_size: int = 4,
    sampling_params: SamplingParams = None,
    use_cot=False,
) -> float:
    # format prompt
    result_dataset = dataset.map(
        lambda x: {"formated_prompt": format_prompt(x, use_cot)},
        input_columns=["problem"],
    )
    # get responses
    start_time = time.time()
    result_dataset = result_dataset.map(
        lambda x: {"llm_response": get_response(x, model, sampling_params)},
        batched=True,
        batch_size=batch_size,
        input_columns="formated_prompt",
    )

    end_time = time.time()
    # extract answer
    result_dataset = result_dataset.map(
        lambda x: {"llm_final_answer": extract_final_answer(x)},
        input_columns=["llm_response"],
    )
    # answer verfier
    result_dataset = result_dataset.map(
        lambda x: {"is_correct": grad_function(x["llm_final_answer"], x["answer"])}
    )
    # calculate model accuracy
    accuracy = calculate_accuracy(result_dataset)
    generation_total_time = end_time - start_time
    return accuracy, generation_total_time

In [ ]:
# run the loop
model_accuracy, generation_time = evaluate(
    llm,
    dataset,
    batch_size=4,
    sampling_params=SamplingParams(temperature=0.1, top_p=0.9),
)
model_accuracy, generation_time

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

...

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

(0.3, 68.05689406394958)

In [24]:
print(f"Model {model_id} accuracy on {datadataset_name}: {model_accuracy:.2f}")
print(f"Generation time: {generation_time:.2f}")

Model Qwen/Qwen3-1.7B-Base accuracy on HuggingFaceH4/MATH-500: 0.50
Generation time: 98.30


# Test Time Scaling 

In [ ]:
# run the loop
cot_model_accuracy, cot_generation_time = evaluate(
    llm,
    dataset,
    batch_size=4,
    sampling_params=SamplingParams(temperature=0.1, top_p=0.9),
    use_cot=True,
)
cot_model_accuracy, cot_generation_time

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

...

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

(0.35, 80.48418736457825)

In [27]:
print(f"Model {model_id} accuracy on {datadataset_name}: {cot_model_accuracy:.2f}")
print(f"Generation time: {generation_time:.2f}")

Model Qwen/Qwen3-1.7B-Base accuracy on HuggingFaceH4/MATH-500: 0.10
Generation time: 98.30


# Self Concistency

In [ ]:
def generate_n_samples(
    model: LLM,
    prompts: [str],
    n: int,
    temp: float,
    top_p: float,
    max_tokens: int = 1024,
):
    sampling_params = SamplingParams(
        n=n, temperature=temp, top_p=top_p, max_tokens=max_tokens
    )
    outputs = model.generate(sampling_params=sampling_params, prompts=prompts)
    return [[out.text for out in outs.outputs] for outs in outputs]

In [ ]:
from collections import Counter


def select_most_common(samples):
    most_common = Counter(samples).most_common(1)
    if most_common:
        return most_common[0][0]
    return samples[0]

In [ ]:
def evaluate(
    model: LLM,
    dataset: Dataset,
    n: int = 4,
    temp: float = 0.7,
    top_p: float = 0.9,
    batch_size: int = 4,
    use_cot=False,
):
    # format prompt
    result_dataset = dataset.map(
        lambda x: {"formated_prompt": format_prompt(x, use_cot)},
        input_columns=["problem"],
    )
    # get responses
    start_time = time.time()
    result_dataset = result_dataset.map(
        lambda x: {
            "n_samples": generate_n_samples(model, x, n=n, temp=temp, top_p=top_p)
        },
        batched=True,
        batch_size=batch_size,
        input_columns="formated_prompt",
    )
    end_time = time.time()
    # extract answer
    result_dataset = result_dataset.map(
        lambda x: {"llm_final_answer": [extract_final_answer(sample) for sample in x]},
        input_columns=["n_samples"],
    )
    # select common
    result_dataset = result_dataset.map(
        lambda x: {"most_common_answer": select_most_common(x)},
        input_columns=["llm_final_answer"],
    )
    # grad answer
    result_dataset = result_dataset.map(
        lambda x: {"is_correct": grad_function(x["most_common_answer"], x["answer"])}
    )
    # calculate model accuracy
    accuracy = calculate_accuracy(result_dataset)
    generation_total_time = end_time - start_time
    return accuracy, generation_total_time

In [ ]:
self_cons_acc, self_cons_time = evaluate(llm, dataset, n=4)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

...

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [115]:
print(f"Model {model_id} accuracy on {datadataset_name}: {self_cons_acc:.2f}")
print(f"Generation time: {self_cons_time:.2f}")

Model Qwen/Qwen3-1.7B-Base accuracy on HuggingFaceH4/MATH-500: 0.50
Generation time: 90.18


In [ ]:
self_cons_acc_cot, self_cons_cot_time = evaluate(llm, dataset, n=4, use_cot=True)

self_cons_acc_cot, self_cons_cot_time

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

(0.7, 94.27057075500488)

# Self Refinement

## Scorer Function

In [18]:
import math
def heuristic_scorer(response:str, boxed_score:float=2.0, length_score:float=500.0, extract_score:float=1.0):
    score = 0.0
    if r"\boxed{" in response:
        score += boxed_score
        answer = extract_final_answer_box(response)
        if answer and answer.strip():
            score += extract_score
    score += 1.5 * math.exp(-len(response)/length_score)
    return score


## log likelikhood

In [19]:
def calculate_logliklihood(model: LLM, prompt: str, response: str) -> float:
    full_input = prompt + response
    tokenizer = model.get_tokenizer()

    full_input_ids = tokenizer.encode(full_input)
    prompt_len = len(tokenizer.encode(prompt))
    response_ids = full_input_ids[prompt_len:]  

    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=1,
        prompt_logprobs=1,
    )
    output = model.generate(prompts=[full_input], sampling_params=sampling_params)
    log_probs = output[0].prompt_logprobs[prompt_len:]

    log_likelihood = 0.0
    for log_prob, actual_token_id in zip(log_probs, response_ids):
        if log_prob is None:
            continue
        if actual_token_id in log_prob:
            log_likelihood += log_prob[actual_token_id].logprob
        else:
            log_likelihood += list(log_prob.values())[-1].logprob

    return log_likelihood#/len(response_ids)


## Format Prompt (Critique, Refine Answer)

In [20]:
def format_critique_prompt(prompt: str, answer: str) -> str:
    critique_prompt = (
        f"Question: {prompt}\n\n"
        f"Draft answer: {answer}\n\n"
        "Write a short crtique, a bullet point fix plan (~ under 120 words).\n\n"
        "Critique:"
    )
    return critique_prompt

In [21]:
def format_refine_prompt(prompt: str, answer: str, critique: str) -> str:
    refine_prompt = (
        "Revise the answer using the critique.\nEnd with a final boxed result: \\boxed{ANSWER}\n\n"
        f"Question: {prompt}\n"
        f"Previous answer: {answer}\n"
        f"Critique: {critique}"
        "\nRevised answer"
    )
    return refine_prompt

## Self Ref Loop

In [22]:
def refinement_loop(model: LLM, scorer_function, is_logprobs:bool, prompt: str, n: int) -> str:
    sampling_params = SamplingParams(temperature=0.1, top_p=0.9, max_tokens=1024)
    response = get_response(prompt, model, sampling_params)[-1]
    response_score = scorer_function(model, prompt, response) if is_logprobs else scorer_function(response)
    for i in range(n):
        critique_prompt = format_critique_prompt(prompt, response)
        critique_sampling_params = SamplingParams(temperature=0.1, top_p=0.9, max_tokens=256)
        critique = get_response(critique_prompt, model, critique_sampling_params)[-1]
        refine_prompt = format_refine_prompt(prompt, response, critique)
        refine_response = get_response(refine_prompt, model, sampling_params)[-1]
        refine_response_score = scorer_function(model, prompt, refine_response) if is_logprobs else scorer_function(response)
        if refine_response_score > response_score:
            response = refine_response
            response_score = refine_response_score
        elif refine_response_score == response_score:
            response = refine_response
            break
        else:
            break
    return response

In [23]:
def evaluate(model: LLM, scorer_function, is_logprobs:bool, dataset: Dataset, use_cot=False, n=3):
    # format prompt
    result_dataset = dataset.map(
        lambda x: {"formated_prompt": format_prompt(x, use_cot)},
        input_columns=["problem"],
    )
    # get responses
    start_time = time.time()
    llm_answers = []
    for row in result_dataset["formated_prompt"]:
        answer = refinement_loop(model, scorer_function, is_logprobs, row, n=n)
        llm_answers.append(answer)
    end_time = time.time()
    result_dataset = result_dataset.add_column("llm_answer", llm_answers)
    # extract answer
    result_dataset = result_dataset.map(
        lambda x: {"llm_final_answer": extract_final_answer(x)},
        input_columns=["llm_answer"],
    )
    # grad answer
    result_dataset = result_dataset.map(
        lambda x: {"is_correct": grad_function(x["llm_final_answer"], x["answer"])}
    )
    # calculate model accuracy
    accuracy = calculate_accuracy(result_dataset)
    generation_total_time = end_time - start_time
    return accuracy, generation_total_time

In [ ]:
self_ref_accuracy, self_ref_time = evaluate(llm, heuristic_scorer, False, dataset, False)

In [77]:
self_ref_accuracy, self_ref_time

(0.45, 363.9998700618744)

In [24]:
self_ref_cot_accuracy, sel_cot_ref_time = evaluate(llm, heuristic_scorer, False, dataset, True)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [25]:
self_ref_cot_accuracy, sel_cot_ref_time

(0.6, 705.8707456588745)

# Others: free the GPU

In [69]:
del llm

In [70]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()